# Customer Twin — Churn Model Training Pipeline

**Goal:** train a Random Forest model that predicts whether an insurance
policyholder will churn (lapse / not renew), using the
`insurance_policyholder_churn_synthetic.csv` dataset as a stand-in for
real Insurise data.

This notebook is written for someone who has **never trained a machine
learning model before**. Every step explains *what* we are doing and
*why*, in plain language, before showing the code. The implementation
itself follows standard, professional scikit-learn practice.

### What you will end up with

At the end of this notebook you will have four files inside a local
`model/` folder, ready to download and drop into the Customer Twin
repository's Risk Intelligence module:

| File | What it is |
|---|---|
| `churn_model.joblib` | The trained Random Forest classifier |
| `preprocessing.joblib` | The fitted scikit-learn preprocessing pipeline (encodes raw features the same way every time) |
| `model_metadata.json` | Everything about how the model was trained: features used, metrics, parameters |
| `feature_schema.json` | The exact input schema the Digital Twin's Feature Mapper needs to build the correct feature vector |

### The mental model

```
Raw Twin features
      │
      ▼
 preprocessing.joblib   (scales numbers, encodes categories)
      │
      ▼
 churn_model.joblib     (Random Forest)
      │
      ▼
 churn probability
```

We build this pipeline once, here, in Colab. The Digital Twin later
loads the two `.joblib` files and reuses them exactly as-is — it never
retrains anything itself in the prototype.

### Two key terms, defined up front

- **X (features):** everything we know about a customer *before* we
  know whether they churned — age, tenure, premium, claims history,
  payment behaviour, etc. This is the input to the model.
- **y (target):** the single column we are trying to predict —
  whether the customer churned (`1`) or was retained (`0`).

Everything from here on is about building `X` correctly (without
cheating) and training a model to predict `y` from it.


## 2. Imports

We only need libraries that ship with Colab by default: `pandas` for
tables, `numpy` for numeric work, `scikit-learn` for the ML pipeline,
`matplotlib` for the one chart we make, and `joblib`/`json` to save
artifacts. Nothing exotic — no deep learning, no XGBoost, no extra
installs required.

In [ ]:
import json
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)


## 3. Load Dataset

**If you are running this in Google Colab:**

1. Click the folder icon on the left sidebar to open the Colab file browser.
2. Click the upload icon and upload:
   - `insurance_policyholder_churn_synthetic.csv`
   - `insurance_policyholder_churn_data_dictionary.csv`
3. Wait for the upload to finish, then run the cell below.

Alternatively, uncomment the `files.upload()` block to get an upload
dialog directly inside the notebook.

In [ ]:
# --- Uncomment this block if you want an in-notebook upload dialog in Colab ---
# from google.colab import files
# uploaded = files.upload()

DATA_PATH = "insurance_policyholder_churn_synthetic.csv"
DICT_PATH = "insurance_policyholder_churn_data_dictionary.csv"

df = pd.read_csv(DATA_PATH)
data_dictionary = pd.read_csv(DICT_PATH)

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")


## 4. Dataset Inspection

Before touching any modelling, we look at the raw data: how big it is,
what types of columns it has, and whether anything looks broken. This
step is purely descriptive — we are not deciding anything about
features yet.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print()
print("Column names:")
print(list(df.columns))


In [ ]:
# Data dictionary, for reference — explains what each raw column means
data_dictionary


In [ ]:
dtype_table = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "n_missing": df.isnull().sum(),
    "pct_missing": (df.isnull().mean() * 100).round(2),
})
dtype_table


In [ ]:
n_duplicated_rows = df.duplicated().sum()
n_duplicated_ids = df["customer_id"].duplicated().sum()

print(f"Fully duplicated rows: {n_duplicated_rows}")
print(f"Duplicated customer_id values: {n_duplicated_ids}")


In [ ]:
categorical_cols_raw = df.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_cols_raw = df.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical-looking columns:", categorical_cols_raw)
print()
print("Numerical-looking columns:", numerical_cols_raw)


**What this tells us:**

- The dataset has 50,000 rows and 40 columns — plenty of data for a
  Random Forest baseline.
- There are no missing values and no duplicated rows or customer IDs,
  which is common for a clean synthetic dataset (real data is rarely
  this tidy).
- `customer_id` is a unique identifier for every row — it must never
  be used as a predictive feature, only as a lookup key.


In [ ]:
target_col = "churn_flag"
print(df[target_col].value_counts())
print()
print(df[target_col].value_counts(normalize=True).round(4))


## 5. Data Quality Analysis

We already confirmed there are no missing values or duplicates. Here
we sanity-check a few numeric ranges and category values to catch
anything implausible (negative amounts, impossible percentages, typos
in category labels).

In [ ]:
numeric_summary = df[numerical_cols_raw].describe().T
numeric_summary


In [ ]:
for col in categorical_cols_raw:
    print(f"--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].value_counts())
    print()


**What this tells us:** the numeric ranges look sensible (ages,
premiums, claim counts, and ratios are all within realistic bounds),
and every categorical column has a small, clean set of values with no
stray typos. The data quality is good enough to move on to
understanding the target.

## 6. Target Variable

According to the data dictionary, `churn_flag` is explicitly described
as: *"Target label: 1 = churned at renewal, 0 = retained"*. This is our
target — we don't need to guess or infer it from other columns.

**In simple language:** we are teaching the model to look at a
customer's profile and answer one yes/no question — *"will this
customer churn?"*

- **`y`** = the `churn_flag` column (what we want to predict)
- **`X`** = every other column that is a legitimate, pre-outcome
  predictor (what we will feed the model) — decided in the next
  section, once we've ruled out leakage.

In [ ]:
print("Target column:", target_col)
print("Target dtype:", df[target_col].dtype)
print()
class_counts = df[target_col].value_counts()
class_pct = df[target_col].value_counts(normalize=True) * 100
target_table = pd.DataFrame({"count": class_counts, "pct": class_pct.round(2)})
target_table


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
df[target_col].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["Retained (0)", "Churned (1)"], rotation=0)
ax.set_ylabel("Number of customers")
ax.set_title("Churn class distribution")
plt.tight_layout()
plt.show()


**Is the target balanced or imbalanced?** About 70% retained vs.
30% churned. This is a **moderate class imbalance** — not extreme, but
enough that plain accuracy could be misleading (a model that always
predicts "retained" would already be ~70% accurate while being
completely useless). Because of this, we will:

- use **stratified** train/test splitting, so both sets keep the same
  70/30 ratio,
- set `class_weight="balanced"` on the Random Forest, and
- look at **precision, recall, F1, and ROC-AUC**, not just accuracy,
  when evaluating the model.


## 7. Data Leakage Analysis

**This is the most important step in the whole notebook.**

Data leakage happens when a feature accidentally gives the model
information it would not actually have *at prediction time* — often
because that feature is itself derived from the outcome, or only
exists *after* the outcome is known. A model trained with leaked
features can look astonishingly accurate and still be completely
useless in production, because the leaking column won't exist yet when
you actually need a prediction.

We go through every column and decide **Keep** or **Remove**, with a
reason — nothing is silently dropped.

In [ ]:
leakage_analysis = [
    ("customer_id", "Remove", "Unique row identifier — carries no predictive signal, only used for lookup/joins."),
    ("as_of_date", "Remove", "Constant across all 50,000 rows (single snapshot date) — contains zero information."),
    ("region_name", "Keep", "Pre-existing customer attribute, known well before renewal."),
    ("age", "Keep", "Pre-existing customer attribute."),
    ("age_band", "Remove", "Deterministic bucketed duplicate of `age` — keeping both adds redundant, perfectly correlated information for no benefit."),
    ("marital_status", "Keep", "Pre-existing customer attribute."),
    ("customer_tenure_months", "Keep", "Known before renewal decision."),
    ("multi_policy_flag", "Keep", "Known before renewal decision."),
    ("num_policies", "Keep", "Known before renewal decision."),
    ("policy_type", "Keep", "Known before renewal decision."),
    ("renewal_month", "Keep", "Known before renewal decision (calendar attribute of the policy)."),
    ("current_premium", "Keep", "Known before renewal decision."),
    ("premium_last_year", "Keep", "Historical, known before renewal decision."),
    ("premium_change_pct", "Keep", "Derived from current vs. last year premium, both known before renewal — legitimate engineered feature, not leakage."),
    ("num_price_increases_last_3y", "Keep", "Historical, known before renewal decision."),
    ("coverage_amount", "Keep", "Known before renewal decision."),
    ("premium_to_coverage_ratio", "Keep", "Derived from two known-in-advance fields — legitimate ratio feature."),
    ("payment_frequency", "Keep", "Known before renewal decision."),
    ("autopay_enabled", "Keep", "Known before renewal decision."),
    ("late_payment_count_12m", "Keep", "Trailing 12-month history, known before renewal decision."),
    ("missed_payment_flag", "Keep", "Deterministic function of `late_payment_count_12m` (per the data dictionary, 1 if >=4 late payments). Redundant with that column but not leakage — both describe pre-renewal payment behaviour, so we keep it and let the model use whichever form it prefers."),
    ("payment_method_change_flag", "Keep", "Trailing history, known before renewal decision."),
    ("num_claims_12m", "Keep", "Trailing 12-month claims history."),
    ("num_approved_claims_12m", "Keep", "Trailing 12-month claims history."),
    ("num_rejected_claims_12m", "Keep", "Trailing 12-month claims history."),
    ("num_pending_claims_12m", "Keep", "Trailing 12-month claims history."),
    ("avg_claim_amount", "Keep", "Trailing 12-month claims history."),
    ("total_claim_amount_12m", "Keep", "Trailing 12-month claims history."),
    ("total_payout_amount_12m", "Keep", "Trailing 12-month claims history."),
    ("payout_ratio_12m", "Keep", "Derived from two trailing-history fields — legitimate ratio feature."),
    ("avg_settlement_time_days", "Keep", "Trailing claims-operations history, known before renewal decision."),
    ("days_since_last_claim", "Keep", "Trailing history, known before renewal decision."),
    ("num_contacts_12m", "Keep", "Trailing customer-service history."),
    ("complaint_flag", "Keep", "Trailing history, known before renewal decision."),
    ("complaint_resolution_days", "Keep", "Trailing history, known before renewal decision."),
    ("quote_requested_flag", "Keep", "Trailing history — a proxy for 'shopping around', known before renewal decision."),
    ("coverage_downgrade_flag", "Keep", "Trailing history, known before renewal decision."),
    ("churn_type", "Remove", "SEVERE LEAKAGE — this is the *reason* the customer churned, which by definition is only known after the churn outcome. It is essentially a restated version of the target."),
    ("churn_probability_true", "Remove", "SEVERE LEAKAGE — the data dictionary states this is the underlying probability used to *generate* the churn label itself. Using it would mean training the model to reverse-engineer its own answer key."),
]

leakage_df = pd.DataFrame(leakage_analysis, columns=["feature", "decision", "reason"])
pd.set_option("display.max_colwidth", None)
leakage_df


**Sanity-check the two severe leakage columns numerically** —
this confirms the reasoning above isn't just theoretical.

In [ ]:
corr_with_target = df["churn_probability_true"].corr(df[target_col])
print(f"Correlation of churn_probability_true with churn_flag: {corr_with_target:.3f}")
print()
print("Mean churn_probability_true when churn_flag == 1:", df.loc[df[target_col] == 1, "churn_probability_true"].mean().round(3))
print("Mean churn_probability_true when churn_flag == 0:", df.loc[df[target_col] == 0, "churn_probability_true"].mean().round(3))
print()
print(pd.crosstab(df["churn_type"], df[target_col]))


**Confirmed:** `churn_probability_true` is strongly correlated
with the target and is *literally the probability used to generate the
label* — this must be removed. `churn_type` is `"No churn"` exactly
when `churn_flag == 0` and a churn reason otherwise — it perfectly
encodes the target and must also be removed. Both are excluded from
the feature set below.

**No uncertainty remains** on any column in this dataset — every
`Keep`/`Remove` decision above is either a pre-renewal customer
attribute (keep) or a post-outcome / identifier / duplicate column
(remove). If you later add real Insurise data with new columns, re-run
this same exercise before trusting any new field.

## 8. Feature Selection

Based on the leakage analysis, here is the final feature list. This
exact list is important beyond this notebook — the Digital Twin's
Feature Mapper must produce a feature vector using **precisely these
column names**, so we also save it later in `feature_schema.json`.

In [ ]:
numerical_features = [
    "age",
    "customer_tenure_months",
    "multi_policy_flag",
    "num_policies",
    "renewal_month",
    "current_premium",
    "premium_last_year",
    "premium_change_pct",
    "num_price_increases_last_3y",
    "coverage_amount",
    "premium_to_coverage_ratio",
    "autopay_enabled",
    "late_payment_count_12m",
    "missed_payment_flag",
    "payment_method_change_flag",
    "num_claims_12m",
    "num_approved_claims_12m",
    "num_rejected_claims_12m",
    "num_pending_claims_12m",
    "avg_claim_amount",
    "total_claim_amount_12m",
    "total_payout_amount_12m",
    "payout_ratio_12m",
    "avg_settlement_time_days",
    "days_since_last_claim",
    "num_contacts_12m",
    "complaint_flag",
    "complaint_resolution_days",
    "quote_requested_flag",
    "coverage_downgrade_flag",
]

categorical_features = [
    "region_name",
    "marital_status",
    "policy_type",
    "payment_frequency",
]

excluded_features = [
    "customer_id",
    "as_of_date",
    "age_band",
    "churn_type",
    "churn_probability_true",
]

feature_columns = numerical_features + categorical_features

print(f"Numerical features ({len(numerical_features)}):")
print(numerical_features)
print()
print(f"Categorical features ({len(categorical_features)}):")
print(categorical_features)
print()
print(f"Excluded ({len(excluded_features)}):")
print(excluded_features)
print()
print(f"TOTAL feature columns used for training: {len(feature_columns)}")


In [ ]:
assert target_col not in feature_columns, "Target must not be in the feature list!"
assert set(feature_columns).issubset(df.columns), "Some feature columns are missing from the dataframe!"
assert len(set(feature_columns)) == len(feature_columns), "Duplicate feature names found!"

X = df[feature_columns].copy()
y = df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)


## 9. Train/Test Split

**In simple language:**

- **Training set (80%)** — the data the model actually learns
  patterns from.
- **Test set (20%)** — data the model *never sees during training*,
  held back purely to check how well it generalizes to new customers.

We split **before** any preprocessing is fitted, and we use
`stratify=y` so that both the training and test sets keep the same
~70/30 churn ratio we saw earlier. `random_state=42` makes the split
reproducible — running this notebook again gives the exact same
split.

In [ ]:
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train):,} ({len(X_train) / len(X):.0%})")
print(f"Test rows:     {len(X_test):,} ({len(X_test) / len(X):.0%})")
print()
print("Training set churn rate:", y_train.mean().round(4))
print("Test set churn rate:    ", y_test.mean().round(4))


**Important rule we follow throughout this notebook:** all
preprocessing (imputers, encoders) is *fitted only on `X_train`*. The
test set is only ever *transformed*, never used to decide how
preprocessing behaves. This prevents the test set from silently
leaking information into training — a common and subtle mistake.

## 10. Preprocessing Pipeline

Random Forests need every input to be numeric, so we need to convert
the categorical columns (like `region_name`) into numbers, and make
sure there's a defined behaviour for any missing values (there are
none in this dataset, but a production pipeline should not assume that
will always stay true — real data will have gaps).

We use scikit-learn's `ColumnTransformer` + `Pipeline` so that:

- numerical columns go through a missing-value imputer (median),
- categorical columns go through a missing-value imputer (most
  frequent value) and then **one-hot encoding** (each category becomes
  its own 0/1 column),
- the *entire* preprocessing step is saved as **one reusable object**
  (`preprocessing.joblib`) — the Digital Twin will call this exact
  same object on new customer data, so training and inference can
  never get out of sync.

In [ ]:
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

# Fit ONLY on the training data
preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Transformed training shape:", X_train_transformed.shape)
print("Transformed test shape:    ", X_test_transformed.shape)


In [ ]:
# The pipeline conceptually is: Raw Twin features -> Preprocessing -> Random Forest -> Prediction
# We keep the fitted `preprocessor` object exactly as-is and reuse it below when we build
# the final combined pipeline (preprocessing + model) that gets saved to disk.

encoded_feature_names = preprocessor.get_feature_names_out()
print(f"Total encoded features after preprocessing: {len(encoded_feature_names)}")
print(encoded_feature_names[:15], "...")


## 11. Random Forest Training

**In simple language:** a Random Forest is a large group ("forest") of
individual decision trees. Each tree looks at the data slightly
differently (a random subset of rows and features) and makes its own
prediction; the forest's final prediction is the average/majority vote
across all trees. This makes it far more stable than any single
decision tree, while still being easy to inspect (via feature
importance, later).

**Parameters used, explained simply:**

- `n_estimators=300` — build 300 trees. More trees generally means a
  more stable prediction, at the cost of more compute; 300 is a solid,
  unremarkable baseline for a dataset this size.
- `max_depth=None` — let each tree grow until its leaves are pure or
  hit `min_samples_leaf`; combined with a large number of trees this
  is a standard, reliable default.
- `min_samples_leaf=5` — a light guard-rail against individual trees
  memorizing tiny quirks of the training data (overfitting).
- `class_weight="balanced"` — because churn is ~30% of the data, this
  tells the forest to weight the minority (churned) class more
  heavily, so it doesn't just learn to always predict "retained".
- `random_state=42` — reproducibility, same as the train/test split.
- `n_jobs=-1` — use all available CPU cores to train faster.

This is intentionally a **baseline MVP model** — we are not doing
extensive hyperparameter tuning, per the project scope.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train_transformed, y_train)
print("Model trained.")
print(model)


## 12. Model Evaluation

Everything below is computed on the **test set only** — data the
model never saw during training. This is the honest measure of how
the model would perform on new customers.

**What each metric means, in plain language:**

- **Accuracy** — % of all predictions that were correct. Can be
  misleading with imbalanced classes (see note below).
- **Precision** — of the customers we *predicted* would churn, what
  fraction actually did? High precision = few false alarms.
- **Recall** — of the customers who *actually* churned, what fraction
  did we catch? High recall = few missed churners.
- **F1-score** — a single number balancing precision and recall.
- **ROC-AUC** — how well the model ranks churners above non-churners
  across all possible decision thresholds; 0.5 = random guessing,
  1.0 = perfect separation.
- **Confusion matrix** — a 2x2 breakdown of correct vs. incorrect
  predictions for each class.

In [ ]:
y_pred = model.predict(X_test_transformed)
y_pred_proba = model.predict_proba(X_test_transformed)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_pred_proba),
}

metrics_df = pd.DataFrame([metrics]).T.rename(columns={0: "value"}).round(4)
metrics_df


**Because churn is imbalanced (~70/30), accuracy alone would be
misleading** — a model that always predicts "retained" would already
score ~70% accuracy while catching zero churners. That's exactly why
we also look at precision, recall, F1, and ROC-AUC above.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual: Retained", "Actual: Churned"],
    columns=["Predicted: Retained", "Predicted: Churned"],
)
cm_df


In [ ]:
print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))


## 13. Check for Suspicious Performance

If the metrics above look near-perfect (very high ROC-AUC, near-1.0
recall and precision at once), that would be a red flag — it usually
means leakage slipped through, or the synthetic data generator baked
in an unrealistically deterministic rule. **We do not celebrate high
numbers automatically — we investigate them.**

A quick way to check: does any single remaining feature correlate
*suspiciously* strongly with the target on its own? A real-world churn
driver should be *informative*, not *deterministic*.

In [ ]:
numeric_corrs = X_train[numerical_features].assign(**{target_col: y_train}).corr()[target_col].drop(target_col)
numeric_corrs = numeric_corrs.sort_values(key=abs, ascending=False)
numeric_corrs.to_frame("correlation_with_churn_flag").head(10)


**Interpretation:** we expect the top correlations here to be
moderate (roughly in the 0.1–0.4 range) — things like missed payments,
complaint activity, or price increases nudging churn risk up, without
any single feature being a near-perfect stand-in for the label. If a
remaining feature showed a correlation above ~0.9, that would call for
going back to the leakage analysis in Section 7 rather than accepting
the result.

Given this is a **synthetic** dataset explicitly built to be "shaped
like" real churn data, model performance here should be read as
**directional and illustrative**, not a guarantee of how the model
will perform once retrained on real Insurise data (this matches the
proxy-data caveat in the Customer Twin design document).

## 14. Feature Importance

Random Forests can report, for each input feature, how much it
contributed to reducing prediction error across all trees. Because
one-hot encoding turns one categorical column (e.g. `region_name`)
into several 0/1 columns (`region_name_Auckland`,
`region_name_Waikato`, ...), we map those back to readable names below
and also roll them back up to the *original* column for a cleaner
view.

**Important caveat:** feature importance tells us which features the
model *relied on most* — it does **not** prove that feature *causes*
churn. It's an association/usage signal, not a causal one.

In [ ]:
importances = model.feature_importances_

importance_df = pd.DataFrame({
    "encoded_feature": encoded_feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)

importance_df.head(15)


In [ ]:
def original_feature_name(encoded_name: str) -> str:
    # Map a ColumnTransformer output name like 'num__age' or
    # 'cat__region_name_Auckland' back to the original raw column name.
    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "", 1)
    if encoded_name.startswith("cat__"):
        remainder = encoded_name.replace("cat__", "", 1)
        for col in categorical_features:
            if remainder.startswith(col):
                return col
    return encoded_name

importance_df["original_feature"] = importance_df["encoded_feature"].apply(original_feature_name)

grouped_importance = (
    importance_df.groupby("original_feature")["importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
grouped_importance


In [ ]:
top_n = 15
top_features = grouped_importance.head(top_n)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_features["original_feature"][::-1], top_features["importance"][::-1], color="#4C72B0")
ax.set_xlabel("Importance (summed across encoded columns)")
ax.set_title(f"Top {top_n} Features by Random Forest Importance")
plt.tight_layout()
plt.show()


This ranking is what feeds the Digital Twin's **Risk
Intelligence** component when it later needs to explain *why* a
customer is flagged as high risk (e.g. surfacing "missed payments" or
"recent price increases" as the drivers behind a recommendation).

## 15. Model Interpretation

For a simple, dependency-free per-customer explanation, we use each
tree's **decision path contribution**, approximated here with the
model's own per-tree predictions spread — this keeps things simple and
avoids installing an extra explainability library (like SHAP) for an
MVP, per the project scope.

A lightweight, robust approach: for one example test customer, we show
their raw feature values next to the **global** top features from
Section 14. This tells a human reviewer *which of this customer's
values fall inside a feature that the model generally weighs heavily*
— a reasonable, simple stand-in for a full per-customer explanation.

**Limitation, stated plainly:** this is global feature importance
applied to one customer's values, not a true per-prediction
attribution (like SHAP values would give). It's a lightweight but
honest MVP-level interpretation.

In [ ]:
example_idx = X_test.index[0]
example_customer = X_test.loc[[example_idx]]
example_true_label = y_test.loc[example_idx]
example_pred_proba = model.predict_proba(preprocessor.transform(example_customer))[0, 1]

print(f"Example customer index: {example_idx}")
print(f"Actual churn_flag: {example_true_label}")
print(f"Predicted churn probability: {example_pred_proba:.3f}")
print()

top_5_features = grouped_importance.head(5)["original_feature"].tolist()
interpretation_table = example_customer[top_5_features].T.rename(columns={example_idx: "customer_value"})
interpretation_table["global_importance_rank"] = range(1, len(top_5_features) + 1)
interpretation_table


## 16. Save Model Artifacts

This is the step that actually connects this notebook to the Customer
Twin repository. We save:

- `model/churn_model.joblib` — the trained Random Forest
- `model/preprocessing.joblib` — the fitted preprocessing pipeline
- `model/model_metadata.json` — everything about how this model was
  trained (no raw customer data included)

We deliberately keep the model and the preprocessing pipeline as two
separate saved objects (rather than one combined `Pipeline`) so the
Digital Twin's Feature Mapper can call `preprocessing.joblib` on its
own — this matches the flow described in the Customer Twin design
document: `Twin State -> Feature Mapper -> preprocessing.joblib ->
churn_model.joblib -> churn probability`.

In [ ]:
import os
os.makedirs("model", exist_ok=True)

MODEL_PATH = "model/churn_model.joblib"
PREPROCESSING_PATH = "model/preprocessing.joblib"
METADATA_PATH = "model/model_metadata.json"
SCHEMA_PATH = "model/feature_schema.json"

joblib.dump(model, MODEL_PATH)
joblib.dump(preprocessor, PREPROCESSING_PATH)

print(f"Saved: {MODEL_PATH}")
print(f"Saved: {PREPROCESSING_PATH}")


In [ ]:
metadata = {
    "model_type": "RandomForestClassifier",
    "target_column": target_col,
    "feature_columns": feature_columns,
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "excluded_features": excluded_features,
    "random_state": RANDOM_STATE,
    "training_row_count": int(len(X_train)),
    "test_row_count": int(len(X_test)),
    "class_distribution": {
        "train": {str(k): int(v) for k, v in y_train.value_counts().to_dict().items()},
        "test": {str(k): int(v) for k, v in y_test.value_counts().to_dict().items()},
    },
    "evaluation_metrics": {k: round(float(v), 4) for k, v in metrics.items()},
    "model_parameters": {
        "n_estimators": model.n_estimators,
        "max_depth": model.max_depth,
        "min_samples_leaf": model.min_samples_leaf,
        "class_weight": "balanced",
        "n_jobs": -1,
    },
    "training_timestamp": datetime.now(timezone.utc).isoformat(),
    "sklearn_version": sklearn.__version__,
    "notes": (
        "Trained on a public/synthetic proxy dataset shaped like Insurise's domain, "
        "not real policyholder records. Treat metrics as directional; re-train on real "
        "Insurise data before production use."
    ),
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {METADATA_PATH}")
print(json.dumps(metadata, indent=2)[:800], "...")


## 17. Reload Artifacts From Disk

To prove the saved files are truly portable (i.e. they will still work
once this Colab session ends and the Digital Twin loads them fresh),
we reload everything from disk into new variable names, rather than
reusing the objects already in memory.

In [ ]:
reloaded_model = joblib.load(MODEL_PATH)
reloaded_preprocessor = joblib.load(PREPROCESSING_PATH)

with open(METADATA_PATH) as f:
    reloaded_metadata = json.load(f)

print("Reloaded model:", type(reloaded_model).__name__)
print("Reloaded preprocessor:", type(reloaded_preprocessor).__name__)
print("Reloaded metadata keys:", list(reloaded_metadata.keys()))


## 18. Test Inference With the Saved Artifacts

Now we run a real prediction using **only the reloaded objects** —
this is exactly what the Digital Twin's inference flow will do:
`Twin State -> preprocessing.joblib -> churn_model.joblib -> churn
probability`.

In [ ]:
test_customer_row_position = 0
test_customer_features = X_test.iloc[[test_customer_row_position]]
test_customer_id = df.loc[test_customer_features.index[0], "customer_id"]

transformed = reloaded_preprocessor.transform(test_customer_features)
predicted_proba = reloaded_model.predict_proba(transformed)[0, 1]
predicted_class = int(predicted_proba >= 0.5)

print("--- Model Prediction (using saved artifacts) ---")
print(f"Customer ID: {test_customer_id}")
print(f"Predicted churn probability: {predicted_proba:.3f}")
print(f"Predicted class: {predicted_class} ({'Churn' if predicted_class == 1 else 'Retained'})")


This confirms the saved `.joblib` files work correctly on their
own, with no dependency on anything still in this notebook's memory —
they are ready to be copied into the Customer Twin repository.

## 19. Export Feature Schema

The Digital Twin's **Feature Mapper** needs to know, for every
expected input feature: its name, its data type, whether it's
numerical or categorical, whether it's required, and — for
categorical features — which values are valid. We build that schema
directly from the training data (never from the test set) so it
reflects exactly what the model was trained on.

In [ ]:
feature_schema = {"features": []}

for col in numerical_features:
    feature_schema["features"].append({
        "name": col,
        "dtype": str(X_train[col].dtype),
        "kind": "numerical",
        "required": True,
        "allowed_values": None,
    })

for col in categorical_features:
    feature_schema["features"].append({
        "name": col,
        "dtype": str(X_train[col].dtype),
        "kind": "categorical",
        "required": True,
        "allowed_values": sorted(X_train[col].dropna().unique().tolist()),
    })

with open(SCHEMA_PATH, "w") as f:
    json.dump(feature_schema, f, indent=2)

print(f"Saved: {SCHEMA_PATH}")
print(f"Total features documented: {len(feature_schema['features'])}")
feature_schema["features"][:3]


## 20. Download These Files

Run the cell below in Google Colab to download all four artifact
files to your computer. Put them into your Customer Twin repository's
model directory:

- `churn_model.joblib`
- `preprocessing.joblib`
- `model_metadata.json`
- `feature_schema.json`

In [ ]:
try:
    from google.colab import files
    for path in [MODEL_PATH, PREPROCESSING_PATH, METADATA_PATH, SCHEMA_PATH]:
        files.download(path)
except ImportError:
    print("Not running in Google Colab — the files are already saved locally in the 'model/' folder:")
    for path in [MODEL_PATH, PREPROCESSING_PATH, METADATA_PATH, SCHEMA_PATH]:
        print(" -", path)


## Final Summary

**What we built:**

- Loaded and inspected a 50,000-row synthetic insurance churn dataset.
- Identified `churn_flag` as the target and, critically, found and
  **removed two severely leaking columns** (`churn_type` and
  `churn_probability_true`) that would otherwise have let the model
  cheat.
- Selected 30 numerical + 4 categorical features as the legitimate,
  pre-outcome feature set (`X`), documented in a leakage table.
- Split the data 80/20 with stratification, fitting all preprocessing
  only on the training set.
- Built a scikit-learn `ColumnTransformer` + `Pipeline` for
  preprocessing (imputation + one-hot encoding).
- Trained a baseline `RandomForestClassifier` (300 trees,
  `class_weight="balanced"`).
- Evaluated honestly on the untouched test set (accuracy, precision,
  recall, F1, ROC-AUC, confusion matrix) and sanity-checked that
  performance isn't suspiciously perfect.
- Extracted and visualized feature importance, mapped back to the
  original (pre-encoding) column names.
- Saved four portable artifacts (`churn_model.joblib`,
  `preprocessing.joblib`, `model_metadata.json`,
  `feature_schema.json`), reloaded them from disk, and confirmed
  inference works end-to-end using only the saved files.

**What you can now tell your advisor:**

- **X** is 34 pre-renewal customer/policy/claims/payment attributes;
  **y** is `churn_flag` (1 = churned, 0 = retained).
- We split the data so the model is judged only on customers it never
  trained on.
- Preprocessing turns raw categories into numeric encoders once, and
  reuses that exact transformation everywhere — training and future
  inference can never drift apart.
- Random Forest predicts churn by combining 300 decision trees trained
  on random subsets of the data.
- We evaluate with precision/recall/F1/ROC-AUC (not just accuracy)
  because the classes are imbalanced.
- Feature importance shows which factors the model leans on most —
  an association signal for Risk Intelligence, not a causal claim.
- The saved model is the direct input to the Customer Twin's Risk
  Intelligence module, following: Twin State -> Feature Mapper ->
  `preprocessing.joblib` -> `churn_model.joblib` -> churn probability.

**Known limitation to flag honestly:** this model is trained on a
**synthetic proxy dataset**, not real Insurise data — per the
project's design document, treat these metrics as directional, and
re-train (`model_version` should be incremented) once real
Claims/CRM/policy data is available.
